# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [ ]:
# Load data
df = pd.read_csv('data/AviationData.csv', low_memory=False)

print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())
df.head()

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [ ]:
# Convert Event.Date to datetime
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

# Filter: accidents only (not incidents)
df = df[df['Investigation.Type'] == 'Accident']

# Filter: professional builds only
df = df[df['Amateur.Built'] == 'No']

# Filter: airplanes only (exclude helicopters, balloons, etc.)
df = df[df['Aircraft.Category'] == 'Airplane']

# Filter: 1983 onwards (max 40-year lifetime)
df = df[df['Event.Date'].dt.year >= 1983]

print("Filtered shape:", df.shape)
df.head()

### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [ ]:
# Fill NaN injury counts with 0 — missing counts are assumed to be zero
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries',
               'Total.Minor.Injuries', 'Total.Uninjured']
df[injury_cols] = df[injury_cols].fillna(0)

# Estimate total passengers on board
df['Total.Passengers'] = df[injury_cols].sum(axis=1)

# Fraction of passengers fatally or seriously injured
# Rows where Total.Passengers == 0 get NaN (no meaningful denominator)
df['Serious.Fatal.Fraction'] = np.where(
    df['Total.Passengers'] > 0,
    (df['Total.Fatal.Injuries'] + df['Total.Serious.Injuries']) / df['Total.Passengers'],
    np.nan
)

print(df[['Total.Passengers', 'Serious.Fatal.Fraction']].describe())
df[['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Passengers', 'Serious.Fatal.Fraction']].head()

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [ ]:
# Inspect Aircraft.damage
print(df['Aircraft.damage'].value_counts(dropna=False))

# Standardize casing
df['Aircraft.damage'] = df['Aircraft.damage'].str.strip().str.title()

# Binary destroyed flag: 1 if Destroyed, 0 otherwise, NaN if unknown
df['Is.Destroyed'] = np.where(
    df['Aircraft.damage'].isna(), np.nan,
    (df['Aircraft.damage'] == 'Destroyed').astype(int)
)

print("\nIs.Destroyed value counts:\n", df['Is.Destroyed'].value_counts(dropna=False))

### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [ ]:
# Standardize Make: strip whitespace, title case
df['Make'] = df['Make'].str.strip().str.title()

# Fix common variants
make_replacements = {
    'Cessna Aircraft': 'Cessna',
    'Cessna Aircraft Company': 'Cessna',
    'Piper Aircraft': 'Piper',
    'Piper Aircraft Inc': 'Piper',
    'Beech': 'Beechcraft',
    'Beech Aircraft': 'Beechcraft',
    'Boeing Commercial Airplanes': 'Boeing',
    'Airbus Industrie': 'Airbus',
    'Mooney Aircraft': 'Mooney',
    'Mooney Aircraft Corp': 'Mooney',
}
df['Make'] = df['Make'].replace(make_replacements)

# Keep only makes with >= 50 occurrences (statistically robust)
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)]

print(f"Remaining makes: {df['Make'].nunique()}")
print(f"Remaining rows: {df.shape[0]}")
df['Make'].value_counts().head(20)

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [ ]:
# Drop rows with missing Model
df = df.dropna(subset=['Model'])

# Standardize Model casing
df['Model'] = df['Model'].str.strip().str.upper()

# Model labels are not unique across makes (e.g. "172" could be Cessna or another)
# Create a unique Make_Model identifier
df['Make_Model'] = df['Make'].str.upper() + '_' + df['Model']

print(f"Unique models: {df['Model'].nunique()}")
print(f"Unique Make_Model combinations: {df['Make_Model'].nunique()}")
df[['Make', 'Model', 'Make_Model']].head()

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [ ]:
# Engine.Type — standardize casing
print("Engine.Type:\n", df['Engine.Type'].value_counts(dropna=False))
df['Engine.Type'] = df['Engine.Type'].str.strip().str.title()

# Weather.Condition — standardize
print("\nWeather.Condition:\n", df['Weather.Condition'].value_counts(dropna=False))
df['Weather.Condition'] = df['Weather.Condition'].str.strip().str.upper()

# Number.of.Engines — coerce to numeric
print("\nNumber.of.Engines:\n", df['Number.of.Engines'].value_counts(dropna=False))
df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce')

# Purpose.of.flight — standardize casing
print("\nPurpose.of.flight:\n", df['Purpose.of.flight'].value_counts(dropna=False))
df['Purpose.of.flight'] = df['Purpose.of.flight'].str.strip().str.title()

# Broad.phase.of.flight — standardize casing
print("\nBroad.phase.of.flight:\n", df['Broad.phase.of.flight'].value_counts(dropna=False))
df['Broad.phase.of.flight'] = df['Broad.phase.of.flight'].str.strip().str.title()

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
# Drop columns where more than 70% of values are NaN
threshold = 0.70
nan_frac = df.isnull().mean()
cols_to_drop = nan_frac[nan_frac > threshold].index.tolist()

print(f"Dropping {len(cols_to_drop)} columns with >{int(threshold*100)}% NaN:\n", cols_to_drop)
df = df.drop(columns=cols_to_drop)

print(f"\nRemaining shape: {df.shape}")

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [ ]:
# Save cleaned DataFrame to CSV for use in analysis notebook
df.to_csv('data/AviationData_Cleaned.csv', index=False)
print("Saved to data/AviationData_Cleaned.csv")
print("Final shape:", df.shape)